In [ ]:
#!/usr/bin/env python3
"""
T5 -- Is ESM3 peptide-conditioning vacuous? (Kaggle, needs ESM3 Forge API.)
Locked threshold, from ../PREREGISTRATION.md:
    PASS: generated-set A-vs-B separation correlates with REAL A-vs-B
    separation at Spearman rho >= 0.4 across >=10 peptide pairs with
    >=20 real TCRs/peptide in the leakage-free train split.
    FAIL: rho < 0.4, or generated distributions statistically
    indistinguishable (p > 0.05) for most pairs.

Credentials
-----------
Requires an EvolutionaryScale Forge API key. DO NOT hardcode it in this
file. Set it as a Kaggle Secret named ESM3_FORGE_API_KEY (Add-ons -> Secrets
in the notebook editor), or as an environment variable of the same name if
running elsewhere. The key is read at runtime only -- never written to disk
or printed.

Pipeline (all implemented below)
---------------------------------
1. Pick >=10 peptide pairs (A, B) from ../data/processed/leakage_free_train.csv
   with >=20 real TCRs each (real_pair_pool()) -- no ESM3 needed for this step.
2. For each peptide, prompt ESM3 (Forge API) to generate N candidate CDR3-beta
   sequences. Conditioning scheme (this project's own choice, documented since
   ESM3 has no published native cross-chain TCR:peptide prompt format):
   construct a single prompt sequence "<peptide><linker><masked CDR3 template>"
   where the masked template length is sampled from the REAL CDR3-beta length
   distribution for that peptide (so length itself isn't leaking the
   separation signal), and let ESM3's generate() fill the masked positions --
   full-sequence context (including the peptide) can influence the fill even
   though this isn't a purpose-built cross-chain conditioning API.
3. Generated-set separation: mean pairwise ESM2 (open, fair-esm, small model --
   used purely as a measurement embedding, not as the generator) embedding
   distance between generated-set(A) and generated-set(B).
4. Real separation: same statistic on REAL CDR3-betas known (VDJdb/IEDB, via
   the leakage-free train split) to bind A vs B.
5. Spearman-correlate generated-separation vs real-separation across all
   pairs; apply the locked bar via write_report().
"""
from __future__ import annotations
import json, os, random
from pathlib import Path
import pandas as pd
import numpy as np

IS_KAGGLE = Path("/kaggle").exists()
REPO = Path("/kaggle/working") if IS_KAGGLE else Path(__file__).resolve().parent.parent
PROC = REPO / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)
OUT_REPORT = PROC / "T5_esm3_conditioning_report.json"

MIN_TCRS_PER_PEPTIDE = 20
MIN_PAIRS = 10
N_GEN_PER_PEPTIDE = 25          # generated CDR3s per peptide -- capped for wall-clock/API-quota
BAR_RHO = 0.4
SEED = 13
LINKER = "GGGGS"                # standard flexible-linker token, used only as a separator

random.seed(SEED)
np.random.seed(SEED)


def _find_train_csv() -> Path:
    candidates = [
        REPO / "data" / "processed" / "leakage_free_train.csv",
        Path("/kaggle/input/datasets/ericdu847/kaggle-t3-gnn-oracle/leakage_free_train.csv"),
    ]
    if IS_KAGGLE:
        for p in Path("/kaggle/input").glob("*/leakage_free_train.csv"):
            candidates.append(p)
    for p in candidates:
        if p.exists():
            print(f"[found] {p}")
            return p
    raise SystemExit(
        "[error] leakage_free_train.csv not found anywhere checked: "
        f"{[str(c) for c in candidates]}. Upload it (and leakage_free_test.csv) "
        "as a Kaggle Dataset attached to this notebook (Add Data -> Upload).")


def _get_forge_api_key() -> str:
    key = os.environ.get("ESM3_FORGE_API_KEY")
    if key:
        return key
    if IS_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            key = UserSecretsClient().get_secret("ESM3_FORGE_API_KEY")
            if key:
                return key
        except Exception:
            pass
    raise SystemExit(
        "[error] ESM3_FORGE_API_KEY not set. Add it as a Kaggle Secret "
        "(notebook editor -> Add-ons -> Secrets -> name it exactly "
        "ESM3_FORGE_API_KEY) and attach it to this notebook, or set the "
        "environment variable of the same name. Never hardcode the key in "
        "this file.")


def real_pair_pool():
    train = pd.read_csv(_find_train_csv())
    counts = train["epitope"].value_counts()
    eligible = counts[counts >= MIN_TCRS_PER_PEPTIDE].index.tolist()
    print(f"[pool] {len(eligible)} epitopes with >= {MIN_TCRS_PER_PEPTIDE} real TCRs")
    return eligible, train


def pick_pairs(eligible: list, n_pairs: int, seed: int = SEED):
    rng = random.Random(seed)
    pool = eligible[:]
    rng.shuffle(pool)
    pairs = []
    i = 0
    while len(pairs) < n_pairs and i + 1 < len(pool):
        pairs.append((pool[i], pool[i + 1]))
        i += 2
    if len(pairs) < n_pairs:
        # allow reuse (documented, per the script's own warning) if the pool
        # can't support n_pairs disjoint pairs
        while len(pairs) < n_pairs:
            a, b = rng.sample(eligible, 2)
            if (a, b) not in pairs and (b, a) not in pairs:
                pairs.append((a, b))
    return pairs


def cdr3_len_distribution(train: pd.DataFrame, peptide: str) -> list:
    lens = train.loc[train["epitope"] == peptide, "cdr3"].str.len().tolist()
    return lens if lens else [14]  # 14aa: typical CDR3-beta length fallback


def generate_esm3_candidates(client, peptide: str, lens: list, n: int) -> list:
    """Generate n CDR3-beta candidates conditioned on `peptide` via the
    documented single-sequence prompting scheme (see module docstring)."""
    from esm.sdk.api import ESMProtein, GenerationConfig
    out = []
    for _ in range(n):
        L = int(np.random.choice(lens))
        template = peptide + LINKER + ("_" * L)  # '_' = masked positions in ESM3's prompt convention
        try:
            protein = ESMProtein(sequence=template)
            cfg = GenerationConfig(track="sequence", num_steps=min(8, max(1, L // 2)), temperature=0.7)
            result = client.generate(protein, cfg)
            full_seq = result.sequence
            cdr3 = full_seq[-L:] if full_seq and len(full_seq) >= L else None
            if cdr3 and all(c.isalpha() for c in cdr3):
                out.append(cdr3.upper())
        except Exception as e:
            print(f"  [warn] ESM3 generation failed for a candidate of {peptide}: {e}")
    return out


_ESM2_MODEL = None
_ESM2_ALPHABET = None
_ESM2_BATCH_CONVERTER = None


def _load_esm2():
    global _ESM2_MODEL, _ESM2_ALPHABET, _ESM2_BATCH_CONVERTER
    if _ESM2_MODEL is not None:
        return
    import torch, esm as fair_esm
    _ESM2_MODEL, _ESM2_ALPHABET = fair_esm.pretrained.esm2_t12_35M_UR50D()
    _ESM2_MODEL.eval()
    if torch.cuda.is_available():
        _ESM2_MODEL = _ESM2_MODEL.cuda()
    _ESM2_BATCH_CONVERTER = _ESM2_ALPHABET.get_batch_converter()


def embed_sequences(seqs: list) -> np.ndarray:
    """Mean-pooled ESM2 embeddings -- a measurement embedding only, not the
    generator. seqs must be non-empty, valid amino-acid strings."""
    import torch
    _load_esm2()
    data = [(str(i), s) for i, s in enumerate(seqs)]
    _, _, tokens = _ESM2_BATCH_CONVERTER(data)
    if torch.cuda.is_available():
        tokens = tokens.cuda()
    with torch.no_grad():
        out = _ESM2_MODEL(tokens, repr_layers=[12])
    reps = out["representations"][12]
    embs = []
    for i, (_, s) in enumerate(data):
        embs.append(reps[i, 1:len(s) + 1].mean(0).cpu().numpy())
    return np.stack(embs)


def mean_pairwise_distance(a: np.ndarray, b: np.ndarray) -> float:
    d = np.linalg.norm(a[:, None, :] - b[None, :, :], axis=-1)
    return float(d.mean())


def separation_stat(emb_a: np.ndarray, emb_b: np.ndarray) -> float:
    """Cross-set mean distance minus the average of the two within-set mean
    distances -- a standard cluster-separation contrast (>0 => sets more
    separated from each other than internally dispersed)."""
    cross = mean_pairwise_distance(emb_a, emb_b)
    within_a = mean_pairwise_distance(emb_a, emb_a) if len(emb_a) > 1 else 0.0
    within_b = mean_pairwise_distance(emb_b, emb_b) if len(emb_b) > 1 else 0.0
    return cross - 0.5 * (within_a + within_b)


def main():
    from esm.sdk.forge import ESM3ForgeInferenceClient

    eligible, train = real_pair_pool()
    if len(eligible) < 2 * MIN_PAIRS:
        print(f"[warn] only {len(eligible)} eligible epitopes -- pairs may reuse epitopes.")

    api_key = _get_forge_api_key()
    client = ESM3ForgeInferenceClient(model="esm3-open", url="https://forge.evolutionaryscale.ai", token=api_key)

    pairs = pick_pairs(eligible, MIN_PAIRS)
    print(f"[pairs] {len(pairs)} peptide pairs selected")

    real_seps, gen_seps, per_pair = [], [], []
    for (pep_a, pep_b) in pairs:
        real_a = train.loc[train["epitope"] == pep_a, "cdr3"].dropna().unique().tolist()
        real_b = train.loc[train["epitope"] == pep_b, "cdr3"].dropna().unique().tolist()
        if len(real_a) < 5 or len(real_b) < 5:
            print(f"  [skip] {pep_a}/{pep_b}: too few real CDR3s after dedup")
            continue

        lens_a = cdr3_len_distribution(train, pep_a)
        lens_b = cdr3_len_distribution(train, pep_b)
        gen_a = generate_esm3_candidates(client, pep_a, lens_a, N_GEN_PER_PEPTIDE)
        gen_b = generate_esm3_candidates(client, pep_b, lens_b, N_GEN_PER_PEPTIDE)
        if len(gen_a) < 5 or len(gen_b) < 5:
            print(f"  [skip] {pep_a}/{pep_b}: too few valid ESM3-generated candidates "
                  f"(got {len(gen_a)}, {len(gen_b)})")
            continue

        emb_real_a = embed_sequences(real_a[:100])
        emb_real_b = embed_sequences(real_b[:100])
        emb_gen_a = embed_sequences(gen_a)
        emb_gen_b = embed_sequences(gen_b)

        real_sep = separation_stat(emb_real_a, emb_real_b)
        gen_sep = separation_stat(emb_gen_a, emb_gen_b)
        real_seps.append(real_sep)
        gen_seps.append(gen_sep)
        per_pair.append({"peptide_a": pep_a, "peptide_b": pep_b,
                          "n_real_a": len(real_a), "n_real_b": len(real_b),
                          "n_gen_a": len(gen_a), "n_gen_b": len(gen_b),
                          "real_separation": real_sep, "generated_separation": gen_sep})
        print(f"  [pair] {pep_a}/{pep_b}: real_sep={real_sep:.4f} gen_sep={gen_sep:.4f}")

    if len(real_seps) < MIN_PAIRS:
        report = {"status": "UNDERPOWERED", "n_usable_pairs": len(real_seps),
                  "min_required": MIN_PAIRS, "per_pair": per_pair,
                  "note": "Too few pairs survived real-data and ESM3-generation "
                          "filters to compute a trustworthy correlation."}
        OUT_REPORT.write_text(json.dumps(report, indent=2))
        print(f"[done] UNDERPOWERED, wrote {OUT_REPORT}")
        return

    write_report(real_seps, gen_seps, per_pair)


def write_report(real_seps, gen_seps, per_pair):
    from scipy.stats import spearmanr
    rho, p = spearmanr(real_seps, gen_seps)
    verdict = "PASS" if (rho >= BAR_RHO and p <= 0.05) else "FAIL"
    report = {
        "n_pairs": len(real_seps),
        "spearman_rho": float(rho),
        "p_value": float(p),
        "bar_rho": BAR_RHO,
        "verdict": verdict,
        "per_pair": per_pair,
    }
    with open(OUT_REPORT, "w") as f:
        json.dump(report, f, indent=2)
    print(f"[done] wrote {OUT_REPORT}: rho={rho:.4f} p={p:.4g} -> {verdict}")


if __name__ == "__main__":
    main()